# Stage 2.2 — Bigram via a neural net

Same task as Stage 2.1, **different mechanic**: instead of counting and normalizing, learn a 27×27 weight matrix by gradient descent. The trained weights should match the count table's log-probabilities up to scale.

**Why bother?** Because the next stages (MLP, BatchNorm, WaveNet) all use this same `inputs → linear → softmax → cross_entropy → backward` loop. Stage 2.2 is where you internalize that loop on the simplest possible model.

## What you'll ship

1. **Training data as tensors**: encode `(prev_char, next_char)` pairs as `(xs, ys)` index tensors
2. **A 27×27 weight matrix `W`** with `requires_grad=True`
3. **Forward**: one-hot encode `xs`, multiply by `W`, softmax → probabilities
4. **Loss**: `F.cross_entropy(logits, ys)` — done in one line, includes softmax
5. **Train loop**: zero grad, backward, step. ~100 iterations at lr=50.
6. **Sample** from the trained model — should look like Stage 2.1's output

**Target NLL**: ≈ 2.46. Same as Stage 2.1 (slightly different due to regularization). If your NLL is way different, something's wrong.

## Reference

Same video as 2.1 — [Karpathy bigram lecture](https://www.youtube.com/watch?v=PaCmpygFfXo), second half.

## Common traps

1. **One-hot encoding via `F.one_hot(xs, num_classes=27).float()`** — needs `.float()` because matmul wants floats. Easy to forget.
2. **`F.cross_entropy(logits, ys)` already includes softmax + log.** Don't softmax the logits before passing them. Double softmax means flat gradient.
3. **lr=50 looks wild but is correct here.** The model is linear and tiny. Karpathy uses lr=50 in the lecture.
4. **Regularization term**: Karpathy adds `0.01 * (W**2).mean()` to the loss. This shrinks W toward 0, equivalent to Laplace smoothing in the count table.

In [ ]:
import torch
import torch.nn.functional as F
from pathlib import Path

words = Path('data/names.txt').read_text().splitlines()
chars = sorted(set(''.join(words)))
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi['.'] = 0
itos = {i: c for c, i in stoi.items()}
V = len(stoi)
print(f'{len(words)} names, vocab={V}')

## Build the training tensors

Flatten every name's bigrams into two long 1-D tensors: `xs` is the input character indices, `ys` is the next-character indices. Shape: `(N,)` where N is the total number of bigrams (~228k).

In [ ]:
# TODO: fill in xs, ys as 1-D int tensors of indices into stoi
# For each name, build chs = ['.'] + list(name) + ['.'], then append (stoi[c1], stoi[c2])
# for each consecutive pair into xs, ys.

xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for a, b in zip(chs, chs[1:]):
        # TODO: append stoi[a] to xs and stoi[b] to ys
        pass

xs = torch.tensor(xs)
ys = torch.tensor(ys)
print(f'xs.shape: {xs.shape}  ys.shape: {ys.shape}')
print(f'first few pairs: xs[:5]={xs[:5].tolist()}  ys[:5]={ys[:5].tolist()}')
# Sanity: there should be ~228k pairs

## The model

A single 27×27 weight matrix. No bias, no nonlinearity. Just `logits = one_hot(xs) @ W`.

In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((V, V), generator=g, requires_grad=True)
print(f'W.shape: {W.shape}  requires_grad: {W.requires_grad}')

## Train loop

Standard PyTorch shape:

1. **Forward**: one-hot `xs` → matmul `W` → `logits`
2. **Loss**: `F.cross_entropy(logits, ys)` + small L2 regularizer on W
3. **Backward**: zero grad, `.backward()`, manual step

After ~100 iters at lr=50, loss should land near 2.46.

In [ ]:
# TODO: write the training loop.
# Steps:
#   xenc = F.one_hot(xs, num_classes=V).float()
#   logits = xenc @ W
#   loss = F.cross_entropy(logits, ys) + 0.01 * (W**2).mean()
#   W.grad = None        # zero grad
#   loss.backward()
#   W.data += -50 * W.grad   # SGD step
#
# Print loss every 10 iters. Expect ~3.7 at init, ~2.46 by iter 100.

for it in range(100):
    # TODO: implement
    pass
    if it % 10 == 0:
        # TODO: print(f'iter {it}  loss {loss.item():.4f}')
        pass

## Sample from the trained model

Same shape as Stage 2.1. The only difference: probabilities come from the softmax of `W[ix]` instead of from a normalized count table.

In [ ]:
# TODO: implement sampler.
# For each name:
#   ix = 0  (start at '.')
#   while True:
#     logits = W[ix]  # this is equivalent to one_hot(ix) @ W, since one_hot picks one row
#     probs = F.softmax(logits, dim=0)
#     ix = torch.multinomial(probs, num_samples=1, generator=g).item()
#     if ix == 0: break
#     out.append(itos[ix])

g = torch.Generator().manual_seed(2147483647)
for _ in range(10):
    # TODO: sample and print one name
    pass

## End-of-stage check

- [ ] Final loss ≈ 2.46
- [ ] Sampled names look similar in shape to Stage 2.1
- [ ] You can articulate **why the count table and the trained `W` produce the same NLL** (hint: they're representing the same conditional distribution, just storing it differently)

One line to `NOTES.md`: what surprised you about replacing counting with gradient descent on the *same* task?

Next: Stage 2.3 — the model gets a hidden layer and the context grows from 1 character to 3. That's where NLL drops to ~2.10.